
# MỨC 2: RANDOM FOREST – HOLD-OUT

## 1. Giới thiệu

**Đề tài:** Dự đoán giá kim cương bằng thuật toán học máy.

**Dữ liệu:** Diamonds Prices2022.csv.

**Thuật toán:** Random Forest Regressor.

**Phương pháp:** Hold-out 80% Train – 20% Test.

**Mục tiêu:** Đánh giá hiệu quả của việc tiền xử lý dữ liệu đối với mô hình Random Forest.

Ở mức 2, nhóm thực hiện:
- Kiểm tra dữ liệu thiếu.
- Loại bỏ các mẫu có kích thước x, y, z không hợp lệ.
- Xóa các dòng dữ liệu trùng lặp.
- Mã hóa các thuộc tính phân loại.

Mô hình vẫn sử dụng các siêu tham số học mặc định, chưa thực hiện tối ưu siêu tham số.


Đọc dữ liệu

In [2]:

# CELL 2: IMPORT CAC THU VIEN CAN THIET

import numpy as np  # Thu vien tinh toan so hoc

import pandas as pd  # Thu vien xu ly du lieu dang bang

from pathlib import Path  # Thu vien xu ly duong dan

from sklearn.model_selection import train_test_split  # Chia du lieu Train/Test

from sklearn.preprocessing import OrdinalEncoder  # Ma hoa thuoc tinh phan loai

from sklearn.ensemble import RandomForestRegressor  # Thuat toan Random Forest

from sklearn.metrics import mean_squared_error  # Tinh MSE

from sklearn.metrics import mean_absolute_error  # Tinh MAE

from sklearn.metrics import r2_score  # Tinh R2

print("Da import cac thu vien thanh cong!")

# CELL 4: DOC DU LIEU GOC

# Duong dan khi notebook chay tu experiments/random_forest
file_path = Path("../../data/raw/Diamonds Prices2022.csv")

# Neu kernel chay tu thu muc goc DiamondPrediction
if not file_path.exists():
    file_path = Path("data/raw/Diamonds Prices2022.csv")

# Bao loi neu khong tim thay tep
if not file_path.exists():
    raise FileNotFoundError("Khong tim thay file Diamonds Prices2022.csv")

# Doc du lieu va dung cot dau tien lam chi so
diamonds = pd.read_csv(file_path, index_col=0)

# In duong dan dang su dung
print("Duong dan:", file_path.resolve())

# In kich thuoc du lieu ban dau
print("Kich thuoc ban dau:", diamonds.shape)

# Dem tong so gia tri thieu
print("So gia tri thieu:", diamonds.isnull().sum().sum())

# Dem so dong trung lap
print("So dong trung lap:", diamonds.duplicated().sum())

# Dem so dong co x, y hoac z khong hop le
invalid_size = (
    (diamonds["x"] <= 0) |
    (diamonds["y"] <= 0) |
    (diamonds["z"] <= 0)
)

# In so dong co kich thuoc khong hop le
print("So dong co kich thuoc khong hop le:", invalid_size.sum())

# Hien thi 5 dong dau tien
display(diamonds.head())


Da import cac thu vien thanh cong!
Duong dan: D:\DiamondPrediction_New\data\raw\Diamonds Prices2022.csv
Kich thuoc ban dau: (53943, 10)
So gia tri thieu: 0
So dong trung lap: 149
So dong co kich thuoc khong hop le: 20


,carat,cut,color,clarity,depth,table,price,x,y,z
1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Tiền xử lý dữ liệu

In [3]:

# CELL 6: TIEN XU LY DU LIEU

# Tao ban sao de giu nguyen du lieu goc
df = diamonds.copy()

# Luu so dong ban dau
initial_rows = len(df)

# Loai bo cac dong co gia tri thieu
df = df.dropna()

# Luu so dong sau khi xu ly gia tri thieu
after_missing = len(df)

# Chi giu cac dong co x, y, z lon hon 0
df = df[
    (df["x"] > 0) &
    (df["y"] > 0) &
    (df["z"] > 0)
]

# Luu so dong sau khi loai kich thuoc khong hop le
after_dimensions = len(df)

# Loai bo cac dong trung lap
df = df.drop_duplicates()

# Dat lai chi so dong tu 0
df = df.reset_index(drop=True)

# In so dong bi loai do thieu du lieu
print("Dong thieu bi loai:", initial_rows - after_missing)

# In so dong bi loai do kich thuoc khong hop le
print("Dong kich thuoc khong hop le bi loai:",
      after_missing - after_dimensions)

# In so dong trung lap bi loai
print("Dong trung lap bi loai:",
      after_dimensions - len(df))

# In so dong con lai
print("So dong sau tien xu ly:", len(df))

# Kiem tra so gia tri thieu con lai
print("Gia tri thieu con lai:", df.isnull().sum().sum())

# Kiem tra so dong trung lap con lai
print("Dong trung lap con lai:", df.duplicated().sum())

# Hien thi 5 dong dau tien sau tien xu ly
display(df.head())


Dong thieu bi loai: 0
Dong kich thuoc khong hop le bi loai: 20
Dong trung lap bi loai: 148
So dong sau tien xu ly: 53775
Gia tri thieu con lai: 0
Dong trung lap con lai: 0


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Chia tập dữ liệu

In [4]:

# CELL 8: CHIA DU LIEU TRAIN VA TEST

# Lay tat ca cac cot tru price lam dac trung dau vao
X = df.drop(columns=["price"])

# Lay cot price lam bien muc tieu
y = df["price"]

# Chia du lieu theo ti le 80% Train va 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X,  # Du lieu dau vao
    y,  # Gia kim cuong
    test_size=0.2,  # 20% danh cho Test
    random_state=42  # Co dinh cach chia
)

# In tong so mau sau tien xu ly
print("Tong so mau:", len(df))

# In so mau Train
print("So mau Train:", len(X_train))

# In so mau Test
print("So mau Test:", len(X_test))


Tong so mau: 53775
So mau Train: 43020
So mau Test: 10755


OrdinalEncoder

In [5]:

# CELL 10: MA HOA DU LIEU PHAN LOAI

# Thu tu mau sac tu thap den cao
color_order = ["J", "I", "H", "G", "F", "E", "D"]

# Thu tu do tinh khiet tu thap den cao
clarity_order = [
    "I1", "SI2", "SI1", "VS2",
    "VS1", "VVS2", "VVS1", "IF"
]

# Thu tu chat luong giac cat tu thap den cao
cut_order = [
    "Fair", "Good", "Very Good",
    "Premium", "Ideal"
]

# Danh sach cac cot can ma hoa
categorical_cols = ["color", "clarity", "cut"]

# Khoi tao bo ma hoa
encoder = OrdinalEncoder(
    categories=[
        color_order,  # Thu tu cot color
        clarity_order,  # Thu tu cot clarity
        cut_order  # Thu tu cot cut
    ]
)

# Tao ban sao cua tap Train
X_train_encoded = X_train.copy()

# Tao ban sao cua tap Test
X_test_encoded = X_test.copy()

# Hoc va ma hoa tren tap Train
X_train_encoded[categorical_cols] = encoder.fit_transform(
    X_train[categorical_cols]
)

# Ma hoa tap Test bang bo ma hoa da hoc
X_test_encoded[categorical_cols] = encoder.transform(
    X_test[categorical_cols]
)

# In so dac trung sau ma hoa
print("So dac trung:", X_train_encoded.shape[1])

# Hien thi du lieu Train sau ma hoa
display(X_train_encoded.head())


So dac trung: 9


,carat,cut,color,clarity,depth,table,x,y,z
41995,0.55,4.0,1.0,3.0,61.6,55.0,5.27,5.29,3.25
10733,1.01,3.0,5.0,2.0,62.0,58.0,6.41,6.37,3.96
22996,1.50,3.0,2.0,4.0,59.9,60.0,7.39,7.44,4.44
11096,0.90,2.0,2.0,4.0,63.0,55.0,6.18,6.13,3.88
35583,0.30,4.0,6.0,3.0,62.4,56.0,4.32,4.27,2.68


Huấn luyện mô hình

In [6]:

# CELL 12: HUAN LUYEN RANDOM FOREST

# Khoi tao Random Forest voi tham so hoc mac dinh
rf_model = RandomForestRegressor(
    random_state=42,  # Co dinh tinh ngau nhien
    n_jobs=-1  # Su dung cac loi CPU kha dung
)

# Huan luyen mo hinh tren tap Train da ma hoa
import time

# Bắt đầu tính thời gian huấn luyện
start_time = time.perf_counter()

# Huấn luyện Random Forest
rf_model.fit(X_train_encoded, y_train)

# Kết thúc tính thời gian huấn luyện
training_time = time.perf_counter() - start_time

# Hiển thị thời gian theo giây
print(f"Thời gian huấn luyện: {training_time:.4f} giây")

# In so cay cua Random Forest
print("So cay:", rf_model.n_estimators)

# Thong bao hoan thanh
print("Da huan luyen Random Forest thanh cong!")


Thời gian huấn luyện: 3.8404 giây
So cay: 100
Da huan luyen Random Forest thanh cong!


Tính MSE, RMSE, MAE, R

In [7]:

# CELL 14: DANH GIA MO HINH TREN TEST

# Du doan gia kim cuong tren tap Test
y_pred = rf_model.predict(X_test_encoded)

# Tinh sai so binh phuong trung binh
mse = mean_squared_error(y_test, y_pred)

# Tinh can bac hai cua MSE
rmse = np.sqrt(mse)

# Tinh sai so tuyet doi trung binh
mae = mean_absolute_error(y_test, y_pred)

# Tinh he so xac dinh R2
r2 = r2_score(y_test, y_pred)

# In ten thuc nghiem
print("MUC 2 - RANDOM FOREST HOLD-OUT 80/20")

# In duong phan cach
print("-" * 45)

# In MSE tren tap Test
print(f"MSE Test : {mse:.4f}")

# In RMSE tren tap Test
print(f"RMSE Test: {rmse:.4f}")

# In MAE tren tap Test
print(f"MAE Test : {mae:.4f}")

# In R2 tren tap Test
print(f"R2 Test  : {r2:.4f}")


MUC 2 - RANDOM FOREST HOLD-OUT 80/20
---------------------------------------------
MSE Test : 272595.0676
RMSE Test: 522.1064
MAE Test : 265.3777
R2 Test  : 0.9828


In [8]:

# BO SUNG BANG DANH GIA TRAIN / TEST
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Du doan tren Train va Test
y_train_pred = rf_model.predict(X_train_encoded)
y_test_pred = rf_model.predict(X_test_encoded)

def tinh_chi_so(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred)
    }

train_metrics = tinh_chi_so(y_train, y_train_pred)
test_metrics = tinh_chi_so(y_test, y_test_pred)

bang_holdout = pd.DataFrame([
    {
        "Tap": "Train",
        "So mau": len(y_train),
        **train_metrics,
        "Thoi gian (giay)": training_time
    },
    {
        "Tap": "Test",
        "So mau": len(y_test),
        **test_metrics,
        "Thoi gian (giay)": training_time
    }
])

print("BANG DANH GIA RANDOM FOREST HOLD-OUT")
display(bang_holdout.round(4))

print("\nTHAM SO MO HINH")
print("n_estimators:", rf_model.n_estimators)
print("max_depth:", rf_model.max_depth)
print("min_samples_split:", rf_model.min_samples_split)
print("min_samples_leaf:", rf_model.min_samples_leaf)
print("max_features:", rf_model.max_features)
print(f"Thoi gian fit: {training_time:.4f} giay")


BANG DANH GIA RANDOM FOREST HOLD-OUT


,Tap,So mau,MAE,MSE,RMSE,R2,Thoi gian (giay)
0,Train,43020,99.6614,42047.5871,205.0551,0.9974,3.8404
1,Test,10755,265.3777,272595.0676,522.1064,0.9828,3.8404



THAM SO MO HINH
n_estimators: 100
max_depth: None
min_samples_split: 2
min_samples_leaf: 1
max_features: 1.0
Thoi gian fit: 3.8404 giay


So sánh mức 1 và mức 2

In [9]:

# CELL 15: SO SANH KET QUA MUC 1 VA MUC 2

# Luu cac chi so Test cua muc 1 da chay
level1_rmse = 533.0261  # RMSE cua muc 1
level1_mae = 261.6055  # MAE cua muc 1
level1_r2 = 0.9818  # R2 cua muc 1

# Tao bang so sanh ket qua hai muc
comparison = pd.DataFrame({
    "Chi so": ["RMSE", "MAE", "R2"],
    "Muc 1": [level1_rmse, level1_mae, level1_r2],
    "Muc 2": [rmse, mae, r2]
})

# Tinh chenh lech Muc 2 tru Muc 1
comparison["Chenh lech"] = comparison["Muc 2"] - comparison["Muc 1"]

# Hien thi bang so sanh
display(comparison.round(4))

# Tinh phan tram thay doi RMSE
rmse_change = (rmse - level1_rmse) / level1_rmse * 100

# In phan tram thay doi RMSE
print(f"Thay doi RMSE: {rmse_change:+.2f}%")

# Neu RMSE giam thi sai so da cai thien
if rmse < level1_rmse:
    print("RMSE Muc 2 thap hon Muc 1.")

# Neu RMSE tang thi sai so chua cai thien
elif rmse > level1_rmse:
    print("RMSE Muc 2 cao hon Muc 1.")

# Neu bang nhau thi sai so khong thay doi
else:
    print("RMSE hai muc bang nhau.")


,Chi so,Muc 1,Muc 2,Chenh lech
0,RMSE,533.0261,522.1064,-10.9197
1,MAE,261.6055,265.3777,3.7722
2,R2,0.9818,0.9828,0.0010


Thay doi RMSE: -2.05%
RMSE Muc 2 thap hon Muc 1.
